In [ ]:
pip install s3fs

In [ ]:
pip install -U aiobotocore==2.13.1 botocore==1.34.106

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import ipywidgets as widgets

# S3 URI to the NetCDF file
# s3_path = (
#     "s3://oidc-rikkert/"
#     "surimi-aggregator/simulations/"
#     "f4b7e204-6541-4dd3-ab39-fd3afc820577/"
#     "f4b7e204-6541-4dd3-ab39-fd3afc820577.nc"
# )

# ds = xr.open_dataset(
#     s3_path,
#     engine="h5netcdf",
#     chunks={"time": 1},
#     backend_kwargs={
#         "storage_options": {
#             "client_kwargs": {
#                 "endpoint_url": "https://minio.dive.edito.eu",
#             },
#             "asynchronous": False,  # ✅ avoids aiobotocore crash
#         }
#     }
# )

# S3 URI to the Zarr directory
s3_path = (
    "s3://oidc-rikkert/"
    "surimi-aggregator/experiments/"
    "b7304e8f-aaf8-4088-a73b-d255fb2a86fa/"
    "zarr"
)

ds = xr.open_zarr(
    s3_path,
    consolidated=True,          # ✅ use if .zmetadata exists
    chunks="auto",              # ✅ Zarr-native chunking
    storage_options={
        "client_kwargs": {
            "endpoint_url": "https://minio.dive.edito.eu",
        },
        "asynchronous": False,   # ✅ avoids aiobotocore issues in notebooks
    },
)

# Inspect dataset
ds



In [ ]:
species_codes = ds["species_code"].astype(str).values.tolist()
fleet_codes = ds["fleet_gear_code"].astype(str).values.tolist()
time_values = [str(t)[:10] for t in ds.time.values]

In [ ]:
species_widget = widgets.Combobox(
    options=species_codes,
    description="Species:",
    ensure_option=True
)

# Select the first species by default
species_widget.value = species_codes[0]

fleet_widget = widgets.Dropdown(
    options=[(code, i) for i, code in enumerate(fleet_codes)],
    description="Fleet:"
)

time_widget = widgets.SelectionSlider(
    options=time_values,
    description="Time:",
    continuous_update=False
)

In [ ]:
def plot_catch(species_code, fleet_idx, time_sel):
    species_idx = species_codes.index(species_code)

    catch = (
        ds["gross_catch"]
        .isel(species=species_idx, fleet=fleet_idx)
        .sel(time=time_sel)
    )
    breakpoint()
    
    plt.figure(figsize=(12, 7))
    ax = plt.axes(projection=ccrs.PlateCarree())

    catch.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap="viridis",
        cbar_kwargs={"label": "Gross catch"}
    )

    ax.coastlines(resolution="10m")
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)

    # Mediterranean extent (adjust if needed)
    ax.set_extent([-2, 9, 36.5, 44.5], crs=ccrs.PlateCarree())

    fleet_name = fleet_codes[fleet_idx]
    ax.set_title(
        f"Gross catch – species {species_code}, fleet {fleet_name}, {time_sel}"
    )

    plt.show()


In [ ]:
widgets.interact(
    plot_catch,
    species_code=species_widget,
    fleet_idx=fleet_widget,
    time_sel=time_widget
)

In [ ]:
out = widgets.Output()

def plot_gross_catch_total(species_code, fleet_idx):
    with out:
        out.clear_output(wait=True)

        # Convert species code to index
        species_idx = species_codes.index(species_code)

        # ✅ Calculate gross_catch_total from gross_catch
        gct = (
#            ds["gross_catch"]     #When you want to calculate the total here instead of the value from  netcdf
            ds["gross_catch_total"]
            .isel(species=species_idx, fleet=fleet_idx)
#            .sum(dim=("lat", "lon"))  # see above
        )

        plt.figure(figsize=(10, 4))
        gct.plot()
        plt.title(
            f"Gross catch total (calculated) – "
            f"species {species_code}, "
            f"fleet {fleet_codes[fleet_idx]}"
        )
        plt.ylabel("Gross catch total")
        plt.xlabel("Time")
        plt.grid(True)
        plt.show()

display(
    widgets.interact(
        plot_gross_catch_total,
        species_code=species_widget,
        fleet_idx=fleet_widget
    ),
    out
)
